## Strategic mind-wandering experiment: eye tracking & thought recalls
Data go to `Data/Strategic_MW_Experiment/` (see `Data/README.md`).
Set `ONE_SUBJECT` below to load a single participant (less RAM), or `None` to load everyone (~1 GB RAM).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
import pandas as pd
sys.path.insert(0, 'Analysis_Codes')
from analysisFunctions import eyeRawKDE, plotRawGaze, plotPupilTimecourse, interactive_pupil_plot

D = 'Data/Strategic_MW_Experiment/'
SCREEN = [0, 0, 1920, 1080]

ONE_SUBJECT = 'ADNA2507'    # None = load all participants (~1 GB RAM)

def read(name):
    """Reads a table; with ONE_SUBJECT set, keeps only that participant (read in chunks to save RAM)."""
    if ONE_SUBJECT is None:
        return pd.read_csv(D + name)
    chunks = pd.read_csv(D + name, chunksize=1_000_000)
    return pd.concat(c[c.Subject == ONE_SUBJECT] for c in chunks).reset_index(drop=True)

behav     = read('behavioral_cleaned.csv')   # one row per trial: MW ratings, recalls, lemmas, story text
gaze      = read('gaze_preproc.csv.gz')      # preprocessed gaze & pupil, 100 Hz
blinks    = read('blinks.csv.gz')
saccades  = read('saccades.csv.gz')
fixations = read('fixations.csv.gz')

print(f"{gaze.Subject.nunique()} participant(s), {len(gaze):,} gaze samples, "
      f"{gaze.memory_usage(deep=True).sum() / 1e9:.2f} GB")

### Choose participant & trial

In [ ]:
subj  = ONE_SUBJECT or 'ADNA2507'
trial = 2

# NOTE: every condition needs its own brackets, otherwise & binds tighter than ==
g = gaze[(gaze.Subject == subj) & (gaze.TrialNum == trial)]

# Events: this participant, this trial, left eye only (every blink/saccade is stored twice: L and R)
b = blinks[(blinks.Subject == subj) & (blinks.TrialNum == trial) & (blinks.Eye == 'L')]
s = saccades[(saccades.Subject == subj) & (saccades.TrialNum == trial) & (saccades.Eye == 'L')]
f = fixations[(fixations.Subject == subj) & (fixations.TrialNum == trial) & (fixations.Eye == 'L')]

info = behav[(behav.Subject == subj) & (behav.TrialNum == trial)].iloc[0]
title = f"{subj} | trial {trial} | {info.Fragment} | MW={info.MW}"
print(title, '|', len(g), 'samples |', len(b), 'blinks |', len(s), 'saccades |', len(f), 'fixations')

### Gaze position & distributions (whole participant)

In [ ]:
fig1 = eyeRawKDE(gaze[gaze.Subject == subj], titl=f'{subj}: gaze & pupil distributions')
fig2 = plotRawGaze(gaze[gaze.Subject == subj], SCREEN, titl=f'{subj}: raw gaze')

### Pupil timecourse (one trial)

In [ ]:
fig3 = plotPupilTimecourse(g, title, b, s, f, chooseViz='011', secChan=1)

### Interactive plot (optional; separate window, needs PyQt5; afterwards run `%matplotlib inline` to go back)

In [ ]:
# %matplotlib qt
# interactive_pupil_plot(ScreenData=g, blinks=b, saccades=s, fixations=f,
#                        gazeCoords=SCREEN, window_size=2000, chooseViz='011')

## Thought recalls: browse participants' utterances
Story fragment | recall | lemmas | MW rating. Dropdown to pick a participant + buttons.

In [ ]:
%matplotlib inline
from storyTools import history_viewer

errors = history_viewer(behav)   # errors -> {Subject: {Fragment: True/False}} ("Mark error" button)

### Recall metrics: length, lexical diversity, self-references, semantic coupling
Needs the NLP stack: `setup_env.ps1 -Nlp` / `bash setup_env.sh --nlp`

In [ ]:
# Load the models ONCE (the sBERT model, ~0.5 GB, is downloaded on first use)
import morfeusz2
from sentence_transformers import SentenceTransformer
from storyTools import utterance_report

morf  = morfeusz2.Morfeusz()
sbert = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

In [ ]:
# Report for the participant/trial chosen above
row = behav[(behav.Subject == subj) & (behav.TrialNum == trial)].iloc[0]
m = utterance_report(row, morf=morf, sbert=sbert)